In [14]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score,accuracy_score


## Load data set

In [15]:
data = pd.read_excel('Students_Dataset.xlsx')


In [16]:
# %pip install openpyxl

In [17]:
# %pip install nltk

In [18]:
data.head()

,Interest_Expression,Predicted_Future_Field
0,Khalid said i enjoy thinking about how structu...,Engineering
1,Jaxson said i enjoy understanding how networks...,Technology/Programming
2,I enjoy understanding how networks connect dev...,Technology/Programming
3,I like learning about future tech and inventions.,Robotics/Innovation
4,I enjoy counting and managing small amounts of...,Business/Finance


In [19]:
data['Predicted_Future_Field'].value_counts()

Predicted_Future_Field
Languages/Literature      1765
Technology/Programming    1721
Sports                    1685
Mechanical/Automotive     1683
Medicine/Healthcare       1674
Art/Creativity            1668
Business/Finance          1656
Robotics/Innovation       1652
Engineering               1633
Science                   1631
Nature/Environment        1628
Social/Leadership         1604
Name: count, dtype: int64

In [20]:
# Input text and target label
x = data["Interest_Expression"]
y = data["Predicted_Future_Field"]

## Text preprocessing


In [24]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    '''
    description:
    this function takes a text as input and apply lowercasing, removing punctuation, tokenization and lammatization on it
    paramters: normal text (str)
    output: cleaned text (str)
    '''
    
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    return " ".join(lemmas)

# Apply preprocessing
texts_cleaned = x.apply(clean_text)



## Save the clean file

In [29]:
cleaned_data = pd.DataFrame({
    'Interest_Expression_cleaned': texts_cleaned,
    'Predicted_Future_Field': y
})

cleaned_data.to_excel("cleaned_Dataset.xlsx", index=False)


In [ ]:
pd.DataFrame(x).head()

,Interest_Expression
0,Khalid said i enjoy thinking about how structu...
1,Jaxson said i enjoy understanding how networks...
2,I enjoy understanding how networks connect dev...
3,I like learning about future tech and inventions.
4,I enjoy counting and managing small amounts of...


## Spliting data into train and testing

In [ ]:
x_train_raw, x_test_raw, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## TDF-IDF MATRIX
we split data after TDF-IDF to avoid ***Data Leakage***

In [ ]:
tfidf = TfidfVectorizer()
x_train = tfidf.fit_transform(x_train_raw)
x_test = tfidf.transform(x_test_raw)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 132939 stored elements and shape (16000, 845)>
  Coords	Values
  (0, 437)	0.13310764831823796
  (0, 457)	0.3588464118131077
  (0, 704)	0.2524595399239601
  (0, 511)	0.43305492510614135
  (0, 400)	0.4139511035320694
  (0, 776)	0.2362426015729449
  (0, 619)	0.43305492510614135
  (0, 526)	0.43305492510614135
  (1, 711)	0.3065924753881775
  (1, 242)	0.1352771841564405
  (1, 572)	0.4553144709147232
  (1, 680)	0.4553144709147232
  (1, 27)	0.1409783663137365
  (1, 814)	0.3618388504660144
  (1, 764)	0.33906501107300147
  (1, 321)	0.4553144709147232
  (2, 437)	0.14335246816054076
  (2, 27)	0.14402702636614884
  (2, 104)	0.5480138676313795
  (2, 660)	0.12202983504958982
  (2, 801)	0.32717663852589257
  (2, 616)	0.31959890050380696
  (2, 1)	0.21762733982173663
  (2, 57)	0.4477448650031874
  (2, 198)	0.4318225012837099
  :	:
  (15996, 500)	0.44053895458899983
  (15997, 242)	0.13833013834296073
  (15997, 27)	0.14415998556705104
  (15997,

## Model
we used the logistic regression model

In [ ]:
model = LogisticRegression(max_iter=500,C=.1)
model.fit(x_train, y_train)

LogisticRegression(C=0.1, max_iter=500)

## Prediction

In [ ]:
y_pred = model.predict(x_test)

## Model evaluation

In [ ]:
acc=accuracy_score(y_test, y_pred)
p=precision_score(y_test, y_pred, average='weighted')
r=recall_score(y_test, y_pred, average='weighted')
f=f1_score(y_test, y_pred, average='weighted')
print(f"Accuracy: {acc}, Precision: {p}, Recall: {r}, F1: {f}")

Accuracy: 0.997, Precision: 0.9970334056993095, Recall: 0.997, F1: 0.9969920323393286


## User prediction

In [ ]:
user_text = input("Enter your interest text: ")

def user_prediction(text):
    # Clean user input
    user_clean = clean_text(user_text)

    # Convert to TF-IDF
    user_vector = tfidf.transform([user_clean])
    
    # Predict future field
    prediction = model.predict(user_vector)
    return prediction[0]

user_prediction(user_text)

'Business/Finance'

## Saving Model

In [ ]:
import pickle

saved_data = {
    "model": model,
    "vectorizer": tfidf
}

with open('student_career_model.pkl', 'wb') as f:
    pickle.dump(saved_data, f)